# Bài 37: Các kiến trúc kinh điển (Classic Architectures)

Bài 34-35 đã xây dựng và huấn luyện một CNN nhỏ. Bài này xem xét các *kiến trúc* CNN đã tiến hóa như thế nào qua khoảng hai thập kỷ, và đi sâu chi tiết vào ý tưởng kiến trúc lớn nhất duy nhất trong lịch sử đó: kết nối dư (residual connection), và bài toán gradient biến mất (vanishing gradient) mà nó được thiết kế để khắc phục.

## ImageNet và ILSVRC: bộ đánh giá chuẩn đã thúc đẩy tất cả (ImageNet and ILSVRC: the benchmark that drove this)
**ImageNet** (<a href="../references.html#deng-2009">Deng et al., 2009</a><span class="landmark-paper">&#9733;</span>) là một *tập dữ liệu*: hơn 14 triệu ảnh, được gán nhãn thủ công trên hơn 20.000 danh mục, được xây dựng bằng cách huy động cộng đồng (crowdsourcing) gán nhãn lên hệ phân cấp danh từ sẵn có của WordNet. Tuy nhiên, chỉ riêng một tập dữ liệu có nhãn lớn không thúc đẩy được tiến bộ &mdash; một *cuộc thi* chung, được chuẩn hóa mới làm được điều đó. Bắt đầu từ năm 2010, **ILSVRC** (ImageNet Large Scale Visual Recognition Challenge — Cuộc thi Nhận dạng Thị giác Quy mô Lớn ImageNet, <a href="../references.html#russakovsky-2015">Russakovsky et al., 2015</a><span class="landmark-paper">&#9733;</span>) đã biến một tập con 1000 danh mục, ~1,2 triệu ảnh của ImageNet thành một bộ đánh giá chuẩn thường niên: mọi nhóm huấn luyện trên cùng dữ liệu và được chấm điểm trên cùng tập kiểm tra được giữ lại, khiến các kiến trúc có thể so sánh trực tiếp, khách quan ở một quy mô chưa ai từng đạt được trước đó. Mọi kiến trúc trong chuyến tham quan bên dưới đều báo cáo kết quả của mình dưới dạng một bài dự thi ILSVRC, và tỉ lệ lỗi top-5 của người thắng cuộc theo từng năm là một đại diện hợp lý cho tiến bộ của lĩnh vực trong thập kỷ mà bài này đề cập:

| Năm | Người thắng | Lỗi top-5 |
|---|---|---|
| 2010 | (đặc trưng thời tiền học sâu) | 28.2% |
| 2012 | AlexNet | 16.4% |
| 2014 | GoogLeNet (VGG bám sát phía sau, ở mức 7.3%) | 6.7% |
| 2015 | ResNet | 3.57% |
| 2017 | SENet | 2.25% |

Trên cùng nhiệm vụ này, một người đã được huấn luyện đạt tỉ lệ lỗi ~5%. ResNet là kiến trúc đầu tiên vượt qua con số này &mdash; đôi khi được gọi là hiệu năng "siêu nhân". Đến năm 2017, với tỉ lệ lỗi của người thắng dưới 2.5%, hạng mục phân loại của ILSVRC đã được cho nghỉ hưu &mdash; về cơ bản nó đã hoàn thành sứ mệnh của một bộ đánh giá chuẩn.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

In [ ]:
years = [2010, 2012, 2014, 2015, 2017]
errors = [28.2, 16.4, 6.7, 3.57, 2.25]
winners = ['(pre-Deep Learning)', 'AlexNet', 'GoogLeNet', 'ResNet', 'SENet']
dxs = [50, 15, 15, 10, 0]  # text location offset 'dx'
dys = [0, 10, 10, 15, 10]  # text location offset 'dy'

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(years, errors, marker='o', color='#1f6fb4', linewidth=2, zorder=3)
for x, y, name, dx, dy in zip(years, errors, winners, dxs, dys):
    ax.annotate(name, (x, y), textcoords='offset points', xytext=(dx, dy), ha='center', fontsize=8.5)
ax.axhline(5.0, color='#d62728', linestyle='--', linewidth=1.2, label='human-level (~5%)')
ax.set_xlabel('year')
ax.set_ylabel('ILSVRC winning top-5 error (%)')
ax.set_title('A decade of progress: ILSVRC winning top-5 error by year')
ax.set_ylim(0, 30)
ax.legend(fontsize=8.5)
plt.show()

## Một chuyến tham quan lịch sử ngắn (A brief historical tour)
Một nhúm kiến trúc đã được phát minh trong lịch sử này, mỗi kiến trúc giải quyết một vấn đề cụ thể:

- **LeNet-5** ("Gradient-Based Learning Applied to Document Recognition", <a href="../references.html#lecun-1998">LeCun et al., 1998</a><span class="landmark-paper">&#9733;</span>) — khuôn mẫu mà Bài 34 đã cài đặt: tích chập, gộp, tích chập, gộp, rồi các lớp kết nối đầy đủ. Được xây dựng để nhận dạng chữ số trên ảnh 32x32.
- **AlexNet** (<a href="../references.html#krizhevsky-2012">Krizhevsky, Sutskever & Hinton, 2012</a><span class="landmark-paper">&#9733;</span>) — cùng khuôn mẫu đó, chỉ là lớn hơn nhiều (5 lớp tích chập, ~60 triệu tham số), được huấn luyện trên ImageNet dùng GPU và ReLU thay vì tanh/sigmoid. Chiến thắng ILSVRC năm 2012 của nó (xem bảng ở trên) là sự kiện bước ngoặt khởi đầu cuộc cách mạng học sâu trong thị giác máy tính &mdash; hồi sinh sự quan tâm của dòng chính đối với mạng nơ-ron sau một mùa đông kéo dài cả thập kỷ (giống "mùa đông AI").
- **VGG** (<a href="../references.html#simonyan-zisserman-2014">Simonyan & Zisserman, 2014</a><span class="landmark-paper">&#9733;</span>) — thay các bộ lọc lớn 11x11/5x5 của AlexNet bằng một chồng sâu các tích chập 3x3 nhỏ. Hai tích chập 3x3 xếp chồng có cùng *trường tiếp nhận (receptive field)* với một tích chập 5x5 (lại là ý tưởng kim tự tháp ở Bài 11) nhưng ít tham số hơn và có thêm một hàm phi tuyến xen giữa.
- **GoogLeNet / Inception** (<a href="../references.html#szegedy-2015">Szegedy et al., 2015</a><span class="landmark-paper">&#9733;</span>) — đi theo chiều rộng thay vì chỉ chiều sâu: mỗi "mô-đun Inception" chạy vài kích thước tích chập (1x1, 3x3, 5x5) song song trên cùng đầu vào và nối (concatenate) các đầu ra của chúng, để mạng không phải cam kết một kích thước bộ lọc cho mỗi lớp. Các tích chập 1x1 được dùng trước để thu nhỏ số kênh một cách rẻ tiền, giữ cho mô-đun rộng hơn vẫn có chi phí chấp nhận được.
- **ResNet** (<a href="../references.html#he-2015-resnet">He et al., 2015</a><span class="landmark-paper">&#9733;</span>) — đẩy độ sâu từ ~19 lớp của VGG lên 50, 101, thậm chí 152, bằng cách giải quyết vấn đề vốn khiến việc xếp chồng thêm lớp một cách ngây thơ trở nên *tệ hơn*, chứ không tốt hơn.
- **SENet** (<a href="../references.html#hu-2018-senet">Hu, Shen & Sun, 2018</a><span class="landmark-paper">&#9733;</span>), người thắng ILSVRC năm 2017 trong bảng ở trên — không phải một hình dạng mạng xương sống (backbone) mới, mà là một sự tinh chỉnh: khối "nén-và-kích-thích (squeeze-and-excitation)" của nó cho phép mỗi lớp học các trọng số theo từng kênh (những kênh đặc trưng nào quan trọng nhất với đầu vào này) thay vì đối xử với mọi kênh như nhau.

Phần còn lại của bài này tập trung vào kết nối dư của ResNet như ý tưởng có hệ quả lớn nhất duy nhất trong lịch sử này.

## Xây dựng một khối ResNet (Building a ResNet block)
### Bài toán gradient biến mất (The vanishing-gradient problem)
Tại sao xếp chồng thêm lớp lại làm các mạng đơn thuần *tệ hơn*? Quy tắc chuỗi của lan truyền ngược nhân một gradient với Jacobian cục bộ của mọi lớp trên đường quay về đầu vào. Nếu các hệ số theo từng lớp đó nhất quán nhỏ hơn 1 — điều dễ xảy ra với một hàm kích hoạt bão hòa như sigmoid, có đạo hàm tối đa là 0.25 — một chồng đủ sâu sẽ nhân gradient với một số rất nhỏ nhiều lần. Gradient tới được các lớp sớm thu nhỏ về 0, và các lớp đó ngừng học hoàn toàn.

Hãy dựng một mạng 30 lớp gồm các lớp `Linear` với hàm kích hoạt sigmoid và theo dõi độ lớn gradient ở mọi độ sâu, cho hai phiên bản: một chồng **đơn thuần (plain)** `x = sigmoid(layer(x))`, và một chồng **dư (residual)** `x = x + 0.3 * sigmoid(layer(x))` trong đó mỗi lớp chỉ phải học một *hiệu chỉnh* nhỏ cộng vào đầu vào của nó, thay vì thay thế hoàn toàn đầu vào.

In [ ]:
DEPTH = 30
WIDTH = 32

def make_layers(seed):
    torch.manual_seed(seed)
    layers = nn.ModuleList([nn.Linear(WIDTH, WIDTH) for _ in range(DEPTH)])
    for layer in layers:
        nn.init.xavier_normal_(layer.weight, gain=nn.init.calculate_gain('sigmoid'))
    return layers

class PlainDeepNet(nn.Module):
    def __init__(self, seed):
        super().__init__()
        self.layers = make_layers(seed)

    def forward(self, x):
        acts = [x]
        for layer in self.layers:
            x = torch.sigmoid(layer(x))
            x.retain_grad()
            acts.append(x)
        return x, acts

class ResidualDeepNet(nn.Module):
    def __init__(self, seed):
        super().__init__()
        self.layers = make_layers(seed)

    def forward(self, x):
        acts = [x]
        for layer in self.layers:
            x = x + 0.3 * torch.sigmoid(layer(x))
            x.retain_grad()
            acts.append(x)
        return x, acts

def grad_norms(model_cls):
    torch.manual_seed(0)
    x = torch.randn(8, WIDTH, requires_grad=True)
    model = model_cls(seed=1)
    out, acts = model(x)
    out.sum().backward()
    return [a.grad.norm().item() for a in acts if a.grad is not None]

plain_norms = grad_norms(PlainDeepNet)
res_norms = grad_norms(ResidualDeepNet)

print(f'plain net:    gradient norm at layer 0 / gradient norm at layer {DEPTH-1} = {plain_norms[0] / plain_norms[-1]:.2e}')
print(f'residual net: gradient norm at layer 0 / gradient norm at layer {DEPTH-1} = {res_norms[0] / res_norms[-1]:.2e}')

plt.figure(figsize=(6, 4))
plt.semilogy(plain_norms, label='plain (sigmoid) net')
plt.semilogy(res_norms, label='residual net')
plt.xlabel('layer depth (0 = input)')
plt.ylabel('gradient norm (log scale)')
plt.title('Gradient magnitude vs. depth')
plt.legend()
plt.show()

Gradient của mạng đơn thuần thu nhỏ khoảng mười chín bậc độ lớn giữa lớp cuối và lớp đầu — về mọi mục đích thực tế, các lớp sớm không nhận được tín hiệu học nào cả. Gradient của mạng dư (ResNet) về cơ bản giữ phẳng qua cả 30 lớp.

Lý do mang tính cấu trúc, không phải do tinh chỉnh: với một kết nối dư, $x_{l+1} = x_l + F(x_l)$, nên $dx_{l+1}/dx_l = I + dF/dx_l$. Lan truyền ngược nhân các Jacobian này với nhau qua các lớp, nhưng mỗi Jacobian đều chứa một ma trận đơn vị $I$ như một số hạng trực tiếp. Điều đó cho gradient một đường đi thẳng về đầu vào mà không bao giờ bị nhân với một đạo hàm sigmoid nhỏ — một lối tắt không bị cản trở, bất kể chồng lớp sâu đến đâu. Mạng đơn thuần không có đường đi như vậy: Jacobian của mọi lớp thuần túy là $dF/dx_l$, nên không có gì ngăn phép nhân lặp lại đẩy tích về 0.

Ảnh động bên dưới diễn lại đúng chiều thực tế của lan truyền ngược: quét ngược từ đầu ra (độ sâu 30) về đầu vào (độ sâu 0), hiển thị chuẩn gradient của mỗi lớp khi nó được tính. Hãy quan sát gradient của ResNet vẫn đủ lớn để hữu ích, trong khi gradient của một mạng truyền thống biến mất về $\approx 0$ khi quá trình quét đến được độ sâu 0.

<img src="../animations/vanishing_gradient_demo.gif" alt="Ảnh động quét chuẩn gradient theo độ sâu lớp, hiển thị từ phải sang trái khớp với chiều thực tế của lan truyền ngược, cho một mạng sigmoid đơn thuần 30 lớp so với một mạng dư: đường cong của mạng đơn thuần giảm khoảng mười chín bậc độ lớn khi tới đầu vào, trong khi đường cong của mạng dư về cơ bản giữ phẳng" style="display: block; margin: 0 auto; max-width: 760px;">

### Chuẩn hóa theo lô (Batch normalization)
ResNet dùng thêm một thành phần nữa: **chuẩn hóa theo lô (batch normalization)** (<a href="../references.html#ioffe-szegedy-2015">Ioffe & Szegedy, 2015</a><span class="landmark-paper">&#9733;</span>). Ý tưởng rất đơn giản &mdash; với mỗi kênh, trừ đi trung bình của kênh đó và chia cho độ lệch chuẩn của nó, tính *trên lô nhỏ hiện tại* (trên các chiều lô, chiều cao và chiều rộng, riêng cho mỗi kênh), để giá trị kích hoạt của mọi kênh luôn có trung bình 0 và độ lệch chuẩn 1 khi đi vào lớp tiếp theo. Hai tham số học được, một hệ số co giãn `gamma` và một độ dịch `beta` theo từng kênh, sau đó được áp dụng bên trên, để lớp vẫn có thể khôi phục một trung bình/tỉ lệ khác nếu điều đó thực sự hữu ích &mdash; chuẩn hóa về 0/1 là một *điểm xuất phát* mà mạng có thể hoàn tác, không phải một ràng buộc cứng.

Các mạng sâu không có chuẩn hóa theo lô dễ gặp một vấn đề liên quan nhưng khác biệt với gradient biến mất: khi huấn luyện cập nhật các lớp sớm, *phân phối* của các giá trị kích hoạt đi vào các lớp sau liên tục dịch chuyển (đôi khi được gọi là "dịch chuyển hiệp biến nội tại — internal covariate shift"), nên các lớp sau liên tục phải đuổi theo một mục tiêu di động. Chuẩn hóa lại ở mọi lớp giữ cho phân phối đó ổn định, điều trên thực tế cho phép dùng tốc độ học cao hơn nhiều và khiến các mạng sâu dễ huấn luyện hơn thấy rõ &mdash; một trong những lý do ResNet có thể đẩy lên 50+ lớp trong khi các kiến trúc trước đó gặp khó khi vượt quá ~20 lớp.

In [ ]:
C = 4
x_bn = torch.randn(8, C, 5, 5)

bn = nn.BatchNorm2d(C)
bn.train()
out_torch = bn(x_bn)

# from scratch: normalize each channel over (batch, height, width), then scale + shift
mean = x_bn.mean(dim=(0, 2, 3), keepdim=True)
var = x_bn.var(dim=(0, 2, 3), unbiased=False, keepdim=True)
x_norm = (x_bn - mean) / torch.sqrt(var + bn.eps)
gamma = bn.weight.view(1, C, 1, 1)
beta = bn.bias.view(1, C, 1, 1)
out_manual = gamma * x_norm + beta

print(f'max abs diff vs nn.BatchNorm2d: {(out_manual - out_torch).abs().max().item():.2e}')
print(f'per-channel mean before: {x_bn.mean(dim=(0, 2, 3)).detach().numpy().round(3)}')
print(f'per-channel std before:  {x_bn.std(dim=(0, 2, 3), unbiased=False).detach().numpy().round(3)}')
print(f'per-channel mean after:  {out_torch.mean(dim=(0, 2, 3)).detach().numpy().round(3)}')
print(f'per-channel std after:   {out_torch.std(dim=(0, 2, 3), unbiased=False).detach().numpy().round(3)}')

Ảnh động bên dưới minh họa phép tính chuẩn hóa theo lô bằng một lưới mẫu (hàng) x kênh (cột). Có ba bước: tính trung bình và độ lệch chuẩn của mỗi kênh (lấy trung bình trên mọi mẫu trong lô, theo từng kênh), chuẩn hóa (theo từng kênh), rồi co giãn và dịch (theo từng kênh) bằng một gamma/beta học được không tầm thường có chủ đích, cho thấy bước cuối không nhất thiết phải khớp với bước đã chuẩn hóa chút nào.

<img src="../animations/batchnorm_demo.gif" alt="Ảnh động phép tính chuẩn hóa theo lô trên một lưới 8 mẫu (hàng) nhân 4 kênh (cột) qua ba bước: bước 1 làm nổi bật mỗi cột với một mũi tên chỉ xuống một hàng bên dưới cho thấy trung bình và độ lệch chuẩn của kênh đó; bước 2 cho thấy công thức (x trừ mu) chia sigma và một mũi tên chỉ lên đưa chính các giá trị trung bình/độ lệch chuẩn đó trở lại lưới, tạo ra các giá trị đã chuẩn hóa; bước 3 cho thấy công thức gamma nhân x-mũ cộng beta với một mũi tên chỉ lên đưa gamma và beta vào lưới, tạo ra các giá trị cuối cùng đã co giãn và dịch" style="display: block; margin: 0 auto; max-width: 700px;">

### Ghép lại: một khối dư tích chập (Putting it together: a convolutional residual block)
Trong một ResNet thực tế, `F` là một cặp tích chập nhỏ (với chuẩn hóa theo lô và ReLU xen giữa), và lối tắt — còn gọi là **kết nối tắt (skip connection)**, vì nó bỏ qua hoàn toàn `F` — cộng *bản đồ đặc trưng đầu vào* trở lại đầu ra của chúng, từng kênh với từng kênh và từng điểm ảnh với từng điểm ảnh:

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return torch.relu(x + out)  # the shortcut: add the block's input back in

block = ResidualBlock(channels=16)
x = torch.randn(4, 16, 20, 20)
y = block(x)
print(f'input shape:  {tuple(x.shape)}')
print(f'output shape: {tuple(y.shape)}  (unchanged — a residual block preserves shape, so blocks can be stacked freely)')

Ảnh động bên dưới đi qua từng bước của cả hai khối: các giai đoạn giống hệt nhau ở cả hai hàng, ngoại trừ kết nối tắt của hàng dư mang `x` thẳng tới phép cộng cuối cùng mà không bị bất kỳ giai đoạn trung gian nào chạm vào.

<img src="../animations/residual_block_demo.gif" alt="Ảnh động so sánh một khối tích chập đơn thuần và một khối dư, đi qua conv1, bn1, relu, conv2, bn2 và một relu cuối cùng ở cả hai hàng: hàng dư còn cho thấy một mũi tên kết nối tắt mang đầu vào thẳng tới một bước cộng ngay trước relu cuối, trong khi hàng đơn thuần nối bn2 trực tiếp tới relu cuối mà không có phép cộng nào" style="display: block; margin: 0 auto; max-width: 820px;">

### Bài tập (Exercises)
1. Trong `PlainDeepNet`/`ResidualDeepNet` ở trên, đổi hàm kích hoạt từ `torch.sigmoid` sang `torch.relu` (đạo hàm của ReLU bằng 1 với mọi đầu vào dương, không bị giới hạn ở 0.25 như sigmoid) và chạy lại phép so sánh chuẩn gradient. Vấn đề biến mất của mạng đơn thuần có tốt lên, tệ đi, hay giữ nguyên?
2. Đổi hệ số co giãn của mạng dư từ `0.3` sang `1.0` (tức là `x + torch.sigmoid(layer(x))` không giảm chấn) và chạy lại. Tỉ số gradient có thay đổi nhiều không? Điều đó gợi ý gì về *lý do* kết nối dư hoạt động — là do hệ số co giãn, hay do chính lối tắt `+ x`?
3. `ResidualBlock` ở trên yêu cầu đầu vào và đầu ra có cùng số kênh, vì chúng được cộng trực tiếp. ResNet thực tế đôi khi cần thay đổi số kênh giữa các khối (ví dụ 16 → 32). Hãy phác thảo (bằng lời, hoặc bằng code) đường lối tắt cần làm gì trong trường hợp đó để phép cộng vẫn có nghĩa.